# Phase 2: GCN and GAT Baselines
Trains GCN and GAT on Twitter15 and Twitter16 with 5 random seeds on a fixed 60/20/20 stratified split.
Reports mean ± std macro-F1 and accuracy on the test set.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import torch

from data.dataset import CascadeDataset
from models.gcn import GCNClassifier
from models.gat import GATClassifier
from training.trainer import run_experiment

DATA_ROOT = "../Twitter15_16_dataset-main"
DEVICE = "cpu"  # MPS kernel overhead is slower than CPU for many small graphs
print(f"Device: {DEVICE}")

In [2]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
IN_CHANNELS = ds15[0].x.shape[1]  # 29
print(f"Twitter15: {len(ds15)} graphs   Twitter16: {len(ds16)} graphs   in_channels: {IN_CHANNELS}")

Twitter15: 1490 graphs   Twitter16: 818 graphs   in_channels: 29


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 200

GCN_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1)
GAT_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, heads=4, dropout=0.1)

In [4]:
print("=" * 50)
print("GCN — Twitter15")
print("=" * 50)
gcn15 = run_experiment(GCNClassifier, GCN_KWARGS, ds15, SEEDS, epochs=EPOCHS, device=DEVICE)

GCN — Twitter15
  seed=0 epoch= 20  train_loss=1.2225  val_f1=0.2434
  seed=0 epoch= 40  train_loss=1.1612  val_f1=0.2468
  seed=0 epoch= 60  train_loss=1.1594  val_f1=0.2627
  seed=0 epoch= 80  train_loss=1.1176  val_f1=0.2602


KeyboardInterrupt: 

In [ ]:
print("=" * 50)
print("GAT — Twitter15")
print("=" * 50)
gat15 = run_experiment(GATClassifier, GAT_KWARGS, ds15, SEEDS, epochs=EPOCHS, device=DEVICE)

In [ ]:
print("=" * 50)
print("GCN — Twitter16")
print("=" * 50)
gcn16 = run_experiment(GCNClassifier, GCN_KWARGS, ds16, SEEDS, epochs=EPOCHS, device=DEVICE)

In [ ]:
print("=" * 50)
print("GAT — Twitter16")
print("=" * 50)
gat16 = run_experiment(GATClassifier, GAT_KWARGS, ds16, SEEDS, epochs=EPOCHS, device=DEVICE)

In [ ]:
print("\n" + "=" * 62)
print(f"{'Model':<10} {'Dataset':<12} {'Acc mean±std':>16} {'Macro-F1 mean±std':>20}")
print("-" * 62)

rows = [
    ("GCN",  "Twitter15", gcn15),
    ("GAT",  "Twitter15", gat15),
    ("GCN",  "Twitter16", gcn16),
    ("GAT",  "Twitter16", gat16),
]
for model, dataset, res in rows:
    acc  = f"{res['test_acc_mean']:.3f} ± {res['test_acc_std']:.3f}"
    f1   = f"{res['test_f1_mean']:.3f} ± {res['test_f1_std']:.3f}"
    print(f"{model:<10} {dataset:<12} {acc:>16} {f1:>20}")

print("=" * 62)
print("(5 seeds, 60/20/20 stratified split, best val-F1 checkpoint)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels  = ["GCN\nTW15", "GAT\nTW15", "GCN\nTW16", "GAT\nTW16"]
means   = [r["test_f1_mean"] for _, _, r in rows]
stds    = [r["test_f1_std"]  for _, _, r in rows]
colors  = ["#7f8c8d", "#3498db", "#7f8c8d", "#3498db"]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels))
bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(0.25, color="#e74c3c", linestyle="--", linewidth=1, label="random chance")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Macro-F1")
ax.set_title("Baseline results (mean ± std, 5 seeds)")
ax.set_ylim(0, 1.0)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{m:.3f}",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("../data/baseline_results.png", dpi=120, bbox_inches="tight")
plt.show()